In [1]:
import numpy as np
import pandas as pd
import random, itertools
from rdkit import Chem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem
from rdkit.ML.Cluster import Butina
from collections import defaultdict

# --------------------------------------------------
# 0) Configuration
# --------------------------------------------------
TEST_FRAC      = 0.2         # target fraction for test set
FP_RADIUS      = 2
FP_NBITS       = 1024
CLUST_THRESH   = 0.35        # Tanimoto threshold for Butina
LABEL_TOL      = 0.1        # allowed deviation of class ratio
SEED           = 42
random.seed(SEED)

In [2]:
# --------------------------------------------------
# 1) Load data and create binary binding labels
# --------------------------------------------------
# Load the primary dataset
data = pd.read_csv('../Input/data_wotherclass.csv')
data = data[['Ikey', 'AC', 'Label', 'SMILES']]
data = data.dropna(subset=["SMILES"]).reset_index(drop=True)

# 1. Filter the DataFrame to keep only the specified labels.
labels_to_keep = ['agonist', 'antagonist', 'nonbinder']
data = data[data['Label'].isin(labels_to_keep)].copy()

# 2. Create the new binary 'Binding' column based on the filtered data.
#    'nonbinder' is 0, and the remaining labels ('agonist', 'antagonist') are 1.
data['Binding'] = np.where(data['Label'] == 'nonbinder', 0, 1)

In [3]:
# --------------------------------------------------
# 2) Scaffold + fingerprint
# --------------------------------------------------
def scaffold_and_fp(smiles):
    """Generates scaffold and fingerprint for a single SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    scaf = MurckoScaffold.MakeScaffoldGeneric(
        MurckoScaffold.GetScaffoldForMol(mol))
    scaf_smiles = Chem.MolToSmiles(scaf, isomericSmiles=False)
    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol, radius=FP_RADIUS, nBits=FP_NBITS)
    return scaf_smiles, fp

# --- Extract unique SMILES ---
print("Step 2: Extracting unique SMILES...")
unique_smiles_df = data[['SMILES']].drop_duplicates().reset_index(drop=True)
n_unique = len(unique_smiles_df)
print(f"Found {n_unique:,} unique SMILES.")

print("Starting FP generation (Serial, 1 process)...")
scaffolds = []
fps = []
for i, smiles in enumerate(unique_smiles_df.SMILES):
    if (i + 1) % 10000 == 0:
        print(f"  ... processing SMILES {i+1:,}/{n_unique:,}")
    scaf, fp = scaffold_and_fp(smiles)
    scaffolds.append(scaf)
    fps.append(fp) # fp is the RDKit ExplicitBitVect object

unique_smiles_df["Scaffold"] = scaffolds
unique_smiles_df["FP"] = fps # Stored as objects in pandas

unique_smiles_df.dropna(subset=["Scaffold", "FP"], inplace=True)
unique_smiles_df.reset_index(drop=True, inplace=True)

n_valid_unique = len(unique_smiles_df)
print(f"FP generation complete. {n_valid_unique:,} valid FPs.")

Step 2: Extracting unique SMILES...
Found 141,134 unique SMILES.
Starting FP generation (Serial, 1 process)...
  ... processing SMILES 10,000/141,134
  ... processing SMILES 20,000/141,134
  ... processing SMILES 30,000/141,134
  ... processing SMILES 40,000/141,134
  ... processing SMILES 50,000/141,134
  ... processing SMILES 60,000/141,134
  ... processing SMILES 70,000/141,134
  ... processing SMILES 80,000/141,134
  ... processing SMILES 90,000/141,134
  ... processing SMILES 100,000/141,134
  ... processing SMILES 110,000/141,134
  ... processing SMILES 120,000/141,134
  ... processing SMILES 130,000/141,134
  ... processing SMILES 140,000/141,134
FP generation complete. 141,134 valid FPs.


In [4]:
# --------------------------------------------------
# 3) Butina clustering on fingerprints
# --------------------------------------------------
def butina_cluster(fp_list, threshold=0.35):
    """
    User's original serial clustering function.
    Calculates and stores the full distance matrix.
    """
    n = len(fp_list)
    print(f"Calculating distance matrix (Serial, 1 process)...")
    print(f"  Molecules (n): {n:,}")
    print(f"  Comparisons (n(n-1)/2): {n*(n-1)//2:,}")
    
    dists = []
    for i in range(1, n):
        sims = DataStructs.BulkTanimotoSimilarity(fp_list[i], fp_list[:i])
        dists.extend([1 - x for x in sims])
        
        if i % 1000 == 0:
            # Provide progress update
            mem_usage = len(dists) * 8 / (1024**3) # Estimate GB
            print(f"  ... processing molecule {i:,}/{n} (Est. Mem: {mem_usage:.2f} GB)")

    print("Distance matrix calculation complete. Starting Butina clustering...")
    clusters = Butina.ClusterData(
        dists, nPts=n, distThresh=1 - threshold, isDistData=True)
    print("Butina clustering complete.")
    return clusters

# --- Run clustering on unique FPs ---
fp_list = unique_smiles_df.FP.tolist()
clusters = butina_cluster(fp_list, CLUST_THRESH) # This will take time.

# --- Map cluster IDs back to unique_smiles_df ---
mol_idx_to_cluster_id = {}
for cid, cluster in enumerate(clusters):
    for idx in cluster:
        mol_idx_to_cluster_id[idx] = cid
unique_smiles_df["Cluster"] = unique_smiles_df.index.map(mol_idx_to_cluster_id)

# --- Map cluster IDs to the ORIGINAL data ---
print("Mapping clusters back to the full dataset...")
smiles_cluster_map = pd.Series(
    unique_smiles_df.Cluster.values, 
    index=unique_smiles_df.SMILES
).to_dict()

data["Cluster"] = data["SMILES"].map(smiles_cluster_map)
data.dropna(subset=["Cluster"], inplace=True) # Handle FPs that failed
data["Cluster"] = data["Cluster"].astype(int)
print("Step 3 complete.")

Calculating distance matrix (Serial, 1 process)...
  Molecules (n): 141,134
  Comparisons (n(n-1)/2): 9,959,332,411
  ... processing molecule 1,000/141134 (Est. Mem: 0.00 GB)
  ... processing molecule 2,000/141134 (Est. Mem: 0.01 GB)
  ... processing molecule 3,000/141134 (Est. Mem: 0.03 GB)
  ... processing molecule 4,000/141134 (Est. Mem: 0.06 GB)
  ... processing molecule 5,000/141134 (Est. Mem: 0.09 GB)
  ... processing molecule 6,000/141134 (Est. Mem: 0.13 GB)
  ... processing molecule 7,000/141134 (Est. Mem: 0.18 GB)
  ... processing molecule 8,000/141134 (Est. Mem: 0.24 GB)
  ... processing molecule 9,000/141134 (Est. Mem: 0.30 GB)
  ... processing molecule 10,000/141134 (Est. Mem: 0.37 GB)
  ... processing molecule 11,000/141134 (Est. Mem: 0.45 GB)
  ... processing molecule 12,000/141134 (Est. Mem: 0.54 GB)
  ... processing molecule 13,000/141134 (Est. Mem: 0.63 GB)
  ... processing molecule 14,000/141134 (Est. Mem: 0.73 GB)
  ... processing molecule 15,000/141134 (Est. Mem: 0.

  ... processing molecule 134,000/141134 (Est. Mem: 66.89 GB)
  ... processing molecule 135,000/141134 (Est. Mem: 67.89 GB)
  ... processing molecule 136,000/141134 (Est. Mem: 68.90 GB)
  ... processing molecule 137,000/141134 (Est. Mem: 69.92 GB)
  ... processing molecule 138,000/141134 (Est. Mem: 70.94 GB)
  ... processing molecule 139,000/141134 (Est. Mem: 71.98 GB)
  ... processing molecule 140,000/141134 (Est. Mem: 73.02 GB)
  ... processing molecule 141,000/141134 (Est. Mem: 74.06 GB)
Distance matrix calculation complete. Starting Butina clustering...
Butina clustering complete.
Mapping clusters back to the full dataset...
Step 3 complete.


In [5]:
# --------------------------------------------------
# 4) Stratified cluster sampling for test set
# --------------------------------------------------
print("Step 4: Starting stratified cluster sampling...")
# This logic is unchanged.
cluster_indices = defaultdict(list)
for data_idx, cluster_id in data["Cluster"].items():
    cluster_indices[cluster_id].append(data_idx)

all_cids = list(cluster_indices.keys())
random.shuffle(all_cids)

target_test = int(len(data) * TEST_FRAC)
test_idx = set()
label_counts = data.Binding.value_counts().to_dict()

def ratio_ok(test_ids):
    if not test_ids:
        return True 
    test_labels = data.loc[list(test_ids), "Binding"].value_counts()
    test_labels_reindexed = test_labels.reindex(label_counts.keys()).fillna(0)
    
    ratio = test_labels_reindexed / len(test_ids)
    base  = (pd.Series(label_counts) / len(data)).reindex(ratio.index).fillna(0)
    
    return ((ratio - base).abs() <= LABEL_TOL).all()

for cid in all_cids:
    cand = set(cluster_indices[cid])
    if len(test_idx) + len(cand) > target_test * 1.05:
        continue
    if ratio_ok(test_idx | cand):
        test_idx.update(cand)
    if len(test_idx) >= target_test:
        break

data["split"] = "train"
data.loc[list(test_idx), "split"] = "test"
print("Step 4 complete.")

Step 4: Starting stratified cluster sampling...
Step 4 complete.


In [6]:
# --------------------------------------------------
# 5) Optional sanity check: inter-set similarity
# --------------------------------------------------
print("Step 5: Running sanity check...")
train_smiles = set(data[data.split == "train"]["SMILES"])
test_smiles  = set(data[data.split == "test"]["SMILES"])

smiles_fp_map = pd.Series(
    unique_smiles_df.FP.values, 
    index=unique_smiles_df.SMILES
).to_dict()

train_fps = [smiles_fp_map[s] for s in train_smiles if s in smiles_fp_map]
test_fps  = [smiles_fp_map[s] for s in test_smiles  if s in smiles_fp_map]

def max_cross_similarity(tfps, tfps2):
    all_max_sims = []
    for i, fp in enumerate(tfps):
        if i % 1000 == 0:
            print(f"Sanity check progress: {i}/{len(tfps)}")
        sims = DataStructs.BulkTanimotoSimilarity(fp, tfps2)
        if sims:
            all_max_sims.append(max(sims))
        else:
            all_max_sims.append(0)
    return all_max_sims

similarities = max_cross_similarity(train_fps, test_fps)

max_sim  = max(similarities)
min_sim  = min(similarities)
mean_sim = np.mean(similarities)

print(f"Max train-test Tanimoto:  {max_sim:.3f}")
print(f"Mean train-test Tanimoto: {mean_sim:.3f}")
print(f"Min train-test Tanimoto:  {min_sim:.3f}")

Step 5: Running sanity check...
Sanity check progress: 0/112621
Sanity check progress: 1000/112621
Sanity check progress: 2000/112621
Sanity check progress: 3000/112621
Sanity check progress: 4000/112621
Sanity check progress: 5000/112621
Sanity check progress: 6000/112621
Sanity check progress: 7000/112621
Sanity check progress: 8000/112621
Sanity check progress: 9000/112621
Sanity check progress: 10000/112621
Sanity check progress: 11000/112621
Sanity check progress: 12000/112621
Sanity check progress: 13000/112621
Sanity check progress: 14000/112621
Sanity check progress: 15000/112621
Sanity check progress: 16000/112621
Sanity check progress: 17000/112621
Sanity check progress: 18000/112621
Sanity check progress: 19000/112621
Sanity check progress: 20000/112621
Sanity check progress: 21000/112621
Sanity check progress: 22000/112621
Sanity check progress: 23000/112621
Sanity check progress: 24000/112621
Sanity check progress: 25000/112621
Sanity check progress: 26000/112621
Sanity ch

In [7]:
# --------------------------------------------------
# 6) Save
# --------------------------------------------------
#cols = ["Ikey", "AC", "Binding", "SMILES"]
data[data.split == "train"].to_csv("../Final/input/train_set_scaf.csv", index=False)
data[data.split == "test"].to_csv("../Final/input/test_set_scaf.csv",  index=False)

print(f"Train: {len(data[data.split=='train']):,} | Test: {len(data[data.split=='test']):,}")

Train: 156,726 | Test: 39,263


In [6]:
from sklearn.model_selection import train_test_split
train_set, valid_set = train_test_split(train_set, test_size=0.2, random_state=SEED, stratify=train_set['Binding'])

In [7]:
train_counts = train_set['Label'].value_counts()
valid_counts = valid_set['Label'].value_counts()
test_counts = test_set['Label'].value_counts()

summary_df = pd.DataFrame({
    'Train': train_counts,
    'Validation': valid_counts,
    'Test': test_counts
})

In [15]:
summary_df

,Train,Validation,Test
antagonist,56707,14248,17779
agonist,40508,10056,11432
nonbinder,28165,7042,10052


In [12]:
cols = ["Ikey", "AC", "Label", "SMILES"]
train_set[cols].to_csv('../Final/input/splits/scaffold_train.csv', index = None)
valid_set[cols].to_csv('../Final/input/splits/scaffold_val.csv', index = None)
test_set[cols].to_csv('../Final/input/splits/scaffold_test.csv', index = None)